<a href="https://colab.research.google.com/github/marcocslima/rag_tests/blob/main/RAG_IA_Operacional.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
caminho_pasta = "/content/drive/MyDrive/tmp/RAG/Documentos_Base"

#SETUP

In [ ]:
!pip install -qU pymupdf4llm langchain langchain-community langchain-huggingface langchain-google-genai faiss-cpu sentence-transformers langchain-text-splitters

In [ ]:
!pip install -qU gradio

In [ ]:
import os
from google.colab import userdata
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr
import glob
import os

In [ ]:
# --- 1. CONFIGURAÇÃO DE AMBIENTE ---
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

# --- 2. INICIALIZAÇÃO DOS COMPONENTES GLOBAIS ---
# Variáveis que serão preenchidas pela função de carga de pasta
db = None
retriever = None
rag_chain = None

print("⚙️ Configurando componentes...")

# Divisor de Texto (Chunking)
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Modelo de Embeddings (Local)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Modelo de Linguagem (Gemini 2.5 Flash)
llm = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview", #"gemini-2.5-flash",
    temperature=0.4
)

# --- 3. DEFINIÇÃO DO PROMPT ANALÍTICO ---
system_prompt = (
    "Você é um Consultor Sênior e Analista de Inteligência de Dados. Sua missão é fornecer respostas profundas, "
    "analíticas e altamente úteis baseadas estritamente nas premissas e informações do contexto fornecido.\n\n"

    "DIRETRIZES DE COMPORTAMENTO (Obrigatório):\n"
    "1. INTELIGÊNCIA ANALÍTICA: Não seja um mero buscador de palavras-chave. Leia nas entrelinhas. "
    "Você tem total liberdade para correlacionar dados, identificar padrões, fazer deduções lógicas e cruzar "
    "informações dentro do contexto para entregar uma resposta completa e inteligente.\n"

    "2. CONHECIMENTO EXTERNO COMO FERRAMENTA: Use sua inteligência e conhecimento geral APENAS de forma instrumental: "
    "para explicar jargões, estruturar raciocínios, realizar cálculos baseados nos dados fornecidos ou traduzir conceitos complexos. "
    "NUNCA use seu conhecimento externo para adicionar FATOS, DADOS ou EVENTOS que não existam no contexto.\n"

    "3. SÍNTESE SEM ALUCINAÇÃO: É permitido extrair consequências lógicas das premissas do texto (ex: se o texto diz que choveu muito, "
    "você pode deduzir que o solo ficou molhado). No entanto, é absolutamente proibido inventar informações para preencher lacunas.\n\n"

    "COMO LIDAR COM INFORMAÇÕES FALTANTES (Não trave):\n"
    "Se a pergunta pedir algo que não está explícito nem pode ser deduzido com segurança do contexto, não encerre a resposta abruptamente. "
    "Siga este protocolo:\n"
    "A) Avise de forma transparente e educada que a informação exata não está disponível nos documentos.\n"
    "B) Imediatamente, ofereça a resposta MAIS PRÓXIMA ou RELEVANTE usando as premissas que você tem. Mostre o que o documento *de fato* diz "
    "sobre o assunto abordado, ajudando o usuário a chegar a uma conclusão aproximada.\n\n"

    "Contexto dos Documentos:\n"
    "```\n{context}\n```"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Função auxiliar para formatar os documentos recuperados
def formatar_docs(documentos):
    return "\n\n".join(doc.page_content for doc in documentos)

print("✅ Infraestrutura pronta! Agora execute a função de carregar a pasta.")

In [ ]:
# Função auxiliar que a IA usa para ler os documentos (definida aqui para evitar o NameError)
def formatar_docs_local(documentos):
    return "\n\n".join(doc.page_content for doc in documentos)

def carregar_pasta(caminho_pasta):
    # Chamamos as variáveis para o escopo global
    global db, retriever, rag_chain, splitter, embeddings, llm, prompt

    # --- PROCESSAMENTO DOS ARQUIVOS ---
    arquivos = glob.glob(os.path.join(caminho_pasta, "*.pdf"))
    if not arquivos:
        print(f"❌ Nenhum PDF encontrado em: {caminho_pasta}")
        return

    docs_list = []
    print(f"📂 Processando {len(arquivos)} arquivos...")

    for f in arquivos:
        nome_f = os.path.basename(f)
        try:
            print(f"📖 Lendo: {nome_f}...", end=" ")
            texto = pymupdf4llm.to_markdown(f)
            chunks = splitter.create_documents([texto], metadatas=[{"fonte": nome_f}])
            docs_list.extend(chunks)
            print("✅")
        except Exception as e:
            print(f"❌ Erro: {e}")

    if not docs_list: return

    # --- CRIAÇÃO DO RAG ---
    print(f"\n🧠 Indexando {len(docs_list)} chunks no FAISS...")
    db = FAISS.from_documents(docs_list, embeddings)
    retriever = db.as_retriever(search_kwargs={"k": 20})

    from langchain_core.runnables import RunnablePassthrough
    from langchain_core.output_parsers import StrOutputParser

    # Função auxiliar local para evitar NameError
    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    rag_chain = (
        {"context": retriever | format_docs, "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    print("🚀 SISTEMA PRONTO PARA O CHAT!")

# Executar a carga automática
carregar_pasta(caminho_pasta)

In [ ]:
# --- Célula para Adicionar Novo Documento ---

# novo_pdf = "/content/declaracao-de-conclusao-da-obra.pdf" # <-- Coloque o caminho do novo arquivo aqui

# if os.path.exists(novo_pdf):
#     print(f"📄 Processando novo documento: {novo_pdf}")

#     # 1. Extrair texto do novo PDF
#     novo_texto_md = pymupdf4llm.to_markdown(novo_pdf)

#     # 2. Dividir em chunks (usando o mesmo splitter da Célula 3)
#     novos_docs = splitter.create_documents([novo_texto_md])
#     print(f"➕ Gerados {len(novos_docs)} novos chunks.")

#     # 3. ADICIONAR ao banco de dados FAISS existente
#     db.add_documents(novos_docs)

#     # 4. Atualizar o retriever para garantir que ele veja os novos dados
#     # Mantemos o k=15 para uma análise profunda
#     retriever = db.as_retriever(search_kwargs={"k": 15})

#     print("✅ Documento adicionado com sucesso ao conhecimento da IA!")
#     print(f"O banco de dados agora possui informações de múltiplos documentos.")
# else:
#     print("❌ Arquivo não encontrado. Verifique se o upload foi feito corretamente.")

#SERVIDOR WEB

In [ ]:
# import gradio as gr

# 1. Função de resposta (mantemos igual, ela recebe a pergunta e o histórico)
def responder_chat(mensagem, historico):
    try:
        # Chamamos o nosso pipeline RAG analítico
        # O invoke do LCEL que montamos espera apenas a string da pergunta
        resposta = rag_chain.invoke(mensagem)
        return resposta
    except Exception as e:
        return f"❌ Erro ao processar consulta: {e}"

# 2. Configurar a interface web (versão simplificada e compatível)
demo = gr.ChatInterface(
    fn=responder_chat,
    title="📊 Chat - RAG",
    description="IA Analítica treinada com os documentos carregados. Faça perguntas técnicas, peça análises de valores ou interpretações de termos contábeis.",
    # Removi os argumentos que estavam causando conflito de versão
)

# 3. Lançar o servidor
# O share=True gera o link público para você abrir no seu navegador fora do Colab
print("⏳ Gerando link de acesso...")
demo.launch(share=True, debug=True)

#CHAT

In [ ]:
# 4ª Célula: Chat Interativo com o PDF
from IPython.display import clear_output

clear_output(wait=True)
print("🤖 Chat Iniciado! Pergunte qualquer coisa sobre o documento.")
print("👉 (Para encerrar, digite 'sair', 'fim' ou 'exit')")
print("-" * 50)

while True:
    # Recebe a pergunta do usuário
    pergunta = input("\n👤 Você: ")

    # Condição para parar o chat
    if pergunta.lower().strip() in ['sair', 'fim', 'exit', 'parar']:
        print("\n👋 Chat encerrado. Até logo!")
        break

    # Ignora se o usuário apertar Enter sem digitar nada
    if not pergunta.strip():
        continue

    print("⏳ Assistente pesquisando no documento...", end="\r")

    try:
        # Envia a pergunta para o nosso RAG (LCEL)
        resposta = rag_chain.invoke(pergunta)

        # Imprime a resposta
        print(" " * 50, end="\r") # Limpa a linha do "pesquisando"
        print(f"🤖 Assistente:\n{resposta}\n")
        print("-" * 50)

    except Exception as e:
        print(f"\n❌ Ocorreu um erro: {e}\n")